# Weighted Fraud Classifier Demo
Demo notebook for presentation

# Imports

In [1]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from IPython.display import display

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


#  Config

Where we adjust the path to our best model.

In [8]:
CONFIG = {
    "data_path": "../data/creditcard.csv",  
    "model_path": "../best_weighted_fraud_model.pt",  
    "seed": 42,
    "input_dim": 30,
    "hidden_dims": [128, 64, 32],
    "dropouts": [0.30, 0.20, 0.10],
    "negative_slope": 0.1,
    "threshold": 0.95
}

# Model

This should match our weighted training notebook.

In [4]:
class FraudDetectionNet(nn.Module):
    def __init__(self, input_features=30, hidden_dims=None, dropouts=None, negative_slope=0.1):
        super(FraudDetectionNet, self).__init__()

        if hidden_dims is None:
            hidden_dims = [128, 64, 32]
        if dropouts is None:
            dropouts = [0.30, 0.20, 0.10]

        assert len(hidden_dims) == len(dropouts), "hidden_dims and dropouts must have same length"

        layers = []
        in_dim = input_features

        for i, (hidden_dim, dropout_rate) in enumerate(zip(hidden_dims, dropouts)):
            layers.append(nn.Linear(in_dim, hidden_dim))

            if i < 2:
                layers.append(nn.BatchNorm1d(hidden_dim))

            layers.append(nn.LeakyReLU(negative_slope))
            layers.append(nn.Dropout(dropout_rate))

            in_dim = hidden_dim

        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# Load dataset

In [5]:
df = pd.read_csv(CONFIG["data_path"])
print(df.shape)
display(df.head())

(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


# Preprocessing

In [6]:
SEED = CONFIG["seed"]
np.random.seed(SEED)
torch.manual_seed(SEED)

df_demo = df.copy()

# If your training notebook used log1p on Time and Amount, keep these
df_demo["Time"] = np.log1p(df_demo["Time"])
df_demo["Amount"] = np.log1p(df_demo["Amount"])

X = df_demo.drop("Class", axis=1).copy()
y = df_demo["Class"].astype(np.float32).values

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=SEED
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

X_test = X_test.astype(np.float32)

print("Test shape:", X_test.shape)
print("Fraud cases in test set:", int(y_test.sum()))

Test shape: (56962, 30)
Fraud cases in test set: 98


# Load trained model

In [9]:
model = FraudDetectionNet(
    input_features=CONFIG["input_dim"],
    hidden_dims=CONFIG["hidden_dims"],
    dropouts=CONFIG["dropouts"],
    negative_slope=CONFIG["negative_slope"]
).to(device)

state_dict = torch.load(CONFIG["model_path"], map_location=device)
model.load_state_dict(state_dict)
model.eval()

print("Model loaded successfully.")
print(model)

Model loaded successfully.
FraudDetectionNet(
  (net): Sequential(
    (0): Linear(in_features=30, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.1)
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): LeakyReLU(negative_slope=0.1)
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): LeakyReLU(negative_slope=0.1)
    (10): Dropout(p=0.1, inplace=False)
    (11): Linear(in_features=32, out_features=1, bias=True)
  )
)


C:\Users\diant\AppData\Local\Temp\ipykernel_30728\1095859410.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(CONFIG["model_path"], map_location=d

# Inference helper

In [10]:
def predict_one(model, sample_array, threshold=0.95):
    """
    sample_array: shape (30,) after preprocessing
    """
    model.eval()
    x = torch.tensor(sample_array, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        logit = model(x).squeeze(1)
        prob = torch.sigmoid(logit).item()
        pred = int(prob >= threshold)

    return {
        "probability_fraud": prob,
        "predicted_label": pred
    }

# Explain prediction helper

In [11]:
def explain_prediction(pred, actual):
    if pred == 1 and actual == 1:
        return "True Positive"
    elif pred == 0 and actual == 0:
        return "True Negative"
    elif pred == 1 and actual == 0:
        return "False Positive"
    else:
        return "False Negative"


def run_demo_samples(indices, threshold=None):
    if threshold is None:
        threshold = CONFIG["threshold"]

    rows = []

    for idx in indices:
        result = predict_one(model, X_test[idx], threshold=threshold)
        actual = int(y_test[idx])

        rows.append({
            "test_index": idx,
            "fraud_probability": round(result["probability_fraud"], 4),
            "predicted_label": result["predicted_label"],
            "actual_label": actual,
            "outcome_type": explain_prediction(result["predicted_label"], actual)
        })

    demo_df = pd.DataFrame(rows)
    display(demo_df)
    return demo_df

# Show some normal cases

In [12]:
normal_indices = np.where(y_test == 0)[0][:5]
run_demo_samples(normal_indices)

,test_index,fraud_probability,predicted_label,actual_label,outcome_type
0,0,0.0007,0,0,True Negative
1,1,0.0000,0,0,True Negative
2,2,0.0000,0,0,True Negative
3,3,0.0013,0,0,True Negative
4,4,0.9448,0,0,True Negative


,test_index,fraud_probability,predicted_label,actual_label,outcome_type
0,0,0.0007,0,0,True Negative
1,1,0.0000,0,0,True Negative
2,2,0.0000,0,0,True Negative
3,3,0.0013,0,0,True Negative
4,4,0.9448,0,0,True Negative


# Show some fraud cases

In [13]:
fraud_indices = np.where(y_test == 1)[0][:5]
run_demo_samples(fraud_indices)

,test_index,fraud_probability,predicted_label,actual_label,outcome_type
0,840,1.0000,1,1,True Positive
1,1146,1.0000,1,1,True Positive
2,3287,1.0000,1,1,True Positive
3,4276,0.9996,1,1,True Positive
4,5077,1.0000,1,1,True Positive


,test_index,fraud_probability,predicted_label,actual_label,outcome_type
0,840,1.0000,1,1,True Positive
1,1146,1.0000,1,1,True Positive
2,3287,1.0000,1,1,True Positive
3,4276,0.9996,1,1,True Positive
4,5077,1.0000,1,1,True Positive


# Mixed demo samples

In [14]:
mixed_indices = list(normal_indices[:3]) + list(fraud_indices[:3])
run_demo_samples(mixed_indices)

,test_index,fraud_probability,predicted_label,actual_label,outcome_type
0,0,0.0007,0,0,True Negative
1,1,0.0000,0,0,True Negative
2,2,0.0000,0,0,True Negative
3,840,1.0000,1,1,True Positive
4,1146,1.0000,1,1,True Positive
5,3287,1.0000,1,1,True Positive


,test_index,fraud_probability,predicted_label,actual_label,outcome_type
0,0,0.0007,0,0,True Negative
1,1,0.0000,0,0,True Negative
2,2,0.0000,0,0,True Negative
3,840,1.0000,1,1,True Positive
4,1146,1.0000,1,1,True Positive
5,3287,1.0000,1,1,True Positive
